In [1]:
# =============================================================================
# GAT-SS — REAL LAB DATA VERSION
# =============================================================================
#
# WHAT CHANGED FROM THE SIMULATED DATA VERSION:
# -----------------------------------------------
# 1. DATA LOADING: simulated data had community_dataset_noGO.csv + community_index.csv
#    (long format: one row per strain per community). Real lab data is WIDE format:
#    one row = one well, columns 1-100 are one-hot strain membership, Lum is the label.
#
# 2. SUPPRESSION LABEL: simulated data gave community_suppression directly.
#    Real lab data gives Lum (already log-normalised luminescence).
#    We compute: suppression = 1 - 10^Lum
#
# 3. NODE FEATURE (ss_luminescence): simulated data had this per-strain per-community.
#    Real lab data: the single-strain file (c_p_SS) gives each strain's Lum.
#    We look up each strain's Lum from that file and use it as the node feature.
#    BUT: the real Lum values are in [-1, 0], while simulated ss_luminescence
#    was in roughly [-10, 0]. So we NORMALISE the node features using StandardScaler
#    so the model sees well-scaled inputs. This is the most important change.
#
# 4. MULTIPLE PLATE FILES: cc_1 to cc_5 are 5 different experimental plates.
#    We pool them all into one dataset (1750 communities total).
#
# 5. COMMUNITY SIZES: fixed set {2,4,6,8,12,16,24,48} instead of 2-50 continuous.
#
# =============================================================================

import numpy as np
import pandas as pd
import random
import os
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as GeoLoader
from torch_geometric.nn import GATConv, global_mean_pool, global_max_pool

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error
from scipy.stats import spearmanr

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

# =============================================================================
# SECTION 0 — SETTINGS
# =============================================================================

SEED       = 42
# ── UPDATE THESE PATHS TO YOUR FOLDER ──
DATA_DIR   = r"C:\Users\Pratham Shah\Downloads\Document dataset"
OUTPUT_DIR = r"C:\Users\Pratham Shah\Downloads\Document dataset\GAT_SS_Realdata"

N_SHAPLEY_PERMUTATIONS = 50
N_EPOCHS   = 300
LR         = 1e-3
BATCH_SIZE = 16
TEST_FRAC  = 0.20

# The 100 strain columns are named "1" through "100" in the lab files
# We define this once and reuse it everywhere
STRAIN_COLS = [str(i) for i in range(1, 101)]

os.makedirs(OUTPUT_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

TEAL   = "#1a6b6b"
ORANGE = "#d4720c"
NAVY   = "#1c2e4a"
RED    = "#E53935"
GREEN  = "#43A047"
BLUE   = "#1E88E5"
GREY   = "#78909C"

plt.rcParams.update({
    "figure.facecolor" : "white",
    "axes.facecolor"   : "white",
    "axes.spines.top"  : False,
    "axes.spines.right": False,
    "font.family"      : "sans-serif",
    "font.size"        : 11,
})

# =============================================================================
# SECTION 1 — LOAD DATA
# =============================================================================
# WHAT IS DIFFERENT:
#   Simulated: community_dataset_noGO.csv has one row per (strain, community)
#              community_index.csv has labels
#   Real lab:  c_p_SS_1_10_4.csv   has one row per strain (monoculture measurement)
#              cc_1_1_10_4.csv to cc_5_1_10_4.csv  each have ~350 community rows
#              Each community row is a one-hot vector: which strains are present
# =============================================================================

print("=" * 65)
print("  GAT-SS REAL LAB DATA — Loading files")
print("=" * 65)

# ── Load single-strain measurements ───────────────────────────────────────
# This file has 100 rows, one per strain.
# Columns 1-100 are one-hot (exactly one 1 per row).
# Lum column = the luminescence of that strain alone with the pathogen.
# This is our node feature — equivalent to ss_luminescence in the simulated data.

ss_df = pd.read_csv(os.path.join(DATA_DIR, "c_p_SS_1_10_4.csv"))

# idxmax(axis=1): for each row, find which column has the maximum value.
# Since it's one-hot (exactly one 1, rest 0), this gives the strain column name.
# .astype(int): convert "72" → 72 so we can do numeric lookups later
ss_df["strain_id"] = ss_df[STRAIN_COLS].idxmax(axis=1).astype(int)

# Build lookup: strain_id (int) → Lum (float)
# This is used when building graphs to look up each strain's node feature
strain_lum_map = dict(zip(ss_df["strain_id"], ss_df["Lum"]))

print(f"  Single-strain strains loaded : {len(ss_df)}")
print(f"  Lum range (raw)              : {ss_df['Lum'].min():.4f} to {ss_df['Lum'].max():.4f}")

# ── Load community data from all 5 plates ─────────────────────────────────
# Each plate file has ~350 rows of communities.
# These are 5 DIFFERENT sets of communities (not replicates of the same ones).
# We pool all 5 into one dataset.

cc_dfs = []
for plate_num in range(1, 6):
    # f-string: f"cc_{plate_num}_1_10_4.csv" builds the filename for each plate
    fname = os.path.join(DATA_DIR, f"cc_{plate_num}_1_10_4.csv")
    df    = pd.read_csv(fname)
    df["plate_replicate"] = plate_num  # track which plate each row came from
    cc_dfs.append(df)

# pd.concat stacks DataFrames vertically. ignore_index=True resets row numbers.
cc_all = pd.concat(cc_dfs, ignore_index=True)

# Count how many strains are in each community
# .sum(axis=1) sums across columns (the 1s in each row = number of strains)
cc_all["n_strains"] = cc_all[STRAIN_COLS].sum(axis=1).astype(int)

# Give each community a unique integer ID
cc_all["community_id"] = range(len(cc_all))

print(f"  Community rows loaded (all 5 plates) : {len(cc_all)}")
print(f"  Community sizes present              : {sorted(cc_all['n_strains'].unique())}")
print(f"  Lum range (raw)                      : {cc_all['Lum'].min():.4f} to {cc_all['Lum'].max():.4f}")

# =============================================================================
# SECTION 2 — COMPUTE SUPPRESSION LABEL
# =============================================================================
# WHAT IS DIFFERENT:
#   Simulated data: community_suppression was pre-computed and given to us.
#   Real lab data:  Lum is log-normalised: Lum = log10(pathogen_comm / pathogen_alone)
#   So: pathogen_fraction = 10^Lum
#   Therefore: suppression = 1 - 10^Lum
#
#   Lum = 0   → suppression = 0.0 (no suppression, pathogen unchanged)
#   Lum = -1  → suppression = 0.9 (90% pathogen killed)
#   Lum = -0.5→ suppression ≈ 0.68
#
#   np.clip(..., 0, 1): clamp to [0,1] to handle tiny floating point noise
# =============================================================================

cc_all["community_suppression"] = np.clip(1.0 - 10.0 ** cc_all["Lum"], 0.0, 1.0)

print(f"\n  Suppression stats:")
print(f"    mean = {cc_all['community_suppression'].mean():.4f}")
print(f"    std  = {cc_all['community_suppression'].std():.4f}")
print(f"    min  = {cc_all['community_suppression'].min():.4f}")
print(f"    max  = {cc_all['community_suppression'].max():.4f}")
print(f"\n  Mean suppression by community size:")
print(cc_all.groupby("n_strains")["community_suppression"].mean().round(3))

# =============================================================================
# SECTION 3 — NORMALISE NODE FEATURES
# =============================================================================
# WHAT IS DIFFERENT (AND WHY THIS IS CRITICAL):
#   Simulated ss_luminescence range: roughly -10 to 0
#   Real lab Lum range:             roughly -1 to 0
#
#   Your GAT-SS model was trained on simulated data where node features
#   had std ≈ 2-3. Real lab features have std ≈ 0.27.
#   If you train fresh on real data, the model will still work fine,
#   but we should normalise to make training stable and comparable.
#
#   StandardScaler: z = (x - mean) / std
#   After scaling: mean = 0, std = 1 — same distribution as a standard normal.
#   This is exactly what we want for neural network inputs.
#
#   IMPORTANT: fit the scaler on the 100 single-strain Lum values,
#   then apply the SAME transformation to the community node features.
#   Do NOT fit on community data because:
#     (a) the same strain appears in many communities (would overweight some)
#     (b) it would leak test set information
# =============================================================================

# Fit scaler on the 100 single-strain Lum values
# reshape(-1, 1): StandardScaler expects a 2D array [n_samples, n_features]
# ss_df["Lum"].values gives a 1D array; reshape makes it [100, 1]
scaler_ss = StandardScaler()
scaler_ss.fit(ss_df["Lum"].values.reshape(-1, 1))

# Apply the scaler to each strain's Lum to get the normalised node feature
# transform() applies: z = (x - learned_mean) / learned_std
# We update strain_lum_map to store the NORMALISED values
for strain_id in strain_lum_map:
    raw_lum = strain_lum_map[strain_id]
    normalised = scaler_ss.transform([[raw_lum]])[0][0]  # [[raw]] → 2D input, [0][0] extracts scalar
    strain_lum_map[strain_id] = normalised

print(f"\n  Normalised node features:")
print(f"    mean = {np.mean(list(strain_lum_map.values())):.4f}  (should be ≈ 0)")
print(f"    std  = {np.std(list(strain_lum_map.values())):.4f}   (should be ≈ 1)")

# =============================================================================
# SECTION 4 — BUILD GRAPHS
# =============================================================================
# WHAT IS DIFFERENT:
#   Simulated: loop over community_ids in community_dataset_noGO.csv,
#              each community has multiple rows (one per member strain).
#   Real lab:  each row in cc_all IS already one complete community.
#              We read the one-hot columns to find which strains are present,
#              then look up each strain's node feature from strain_lum_map.
#   Everything else (edge construction, Data object, labels) is IDENTICAL.
# =============================================================================

print("\n  Building graphs ...")

graph_list = []

# iterrows(): iterate over DataFrame rows as (index, Series) pairs.
# This is slower than vectorised ops but fine here since we need per-row logic.
for _, row in cc_all.iterrows():

    # Find which strains are in this community.
    # row[STRAIN_COLS].values gives array of 0s and 1s (length 100).
    # enumerate gives (position, value) pairs.
    # We collect the strain column name (as int) wherever value == 1.
    present_strains = [
        int(STRAIN_COLS[i])
        for i, v in enumerate(row[STRAIN_COLS].values)
        if v == 1
    ]

    n_nodes = len(present_strains)

    # Build node feature matrix: [n_nodes, 1]
    # Each node gets the normalised Lum of that strain from single-strain file.
    node_features = []
    for s in present_strains:
        lum = strain_lum_map.get(s, 0.0)  # .get(key, default): returns 0 if strain not found
        node_features.append([lum])        # shape [1] per node, becomes [n_nodes, 1] total

    x = torch.tensor(node_features, dtype=torch.float)  # [n_nodes, 1]

    # Fully connected edges (no self-loops) — UNCHANGED from simulated version
    src, dst = [], []
    for i in range(n_nodes):
        for j in range(n_nodes):
            if i != j:
                src.append(i)
                dst.append(j)
    edge_index = torch.tensor([src, dst], dtype=torch.long)  # [2, n*(n-1)]

    # Label: suppression score we computed above
    y = torch.tensor([float(row["community_suppression"])], dtype=torch.float)

    # Create the PyG Data object — UNCHANGED
    data = Data(x=x, edge_index=edge_index, y=y)

    # Store metadata (not used by GNN, useful for analysis)
    data.community_id    = int(row["community_id"])
    data.community_size  = n_nodes
    data.strain_ids      = present_strains
    data.plate_replicate = int(row["plate_replicate"])
    data.lum_raw         = float(row["Lum"])

    graph_list.append(data)

print(f"  Graphs built: {len(graph_list)}")
sample = graph_list[0]
print(f"  Sample graph — nodes: {sample.x.shape[0]}, "
      f"features per node: {sample.x.shape[1]}, "
      f"edges: {sample.edge_index.shape[1]}")
print(f"  Sample node features (first 3): {sample.x[:3, 0].tolist()}")
print(f"  Sample label: {sample.y.item():.4f}")

# =============================================================================
# SECTION 5 — TRAIN / TEST SPLIT
# =============================================================================
# WHAT IS DIFFERENT:
#   We add stratified splitting by community size.
#   The community sizes are: {2,4,6,8,12,16,24,48}
#   Without stratification, random splitting might put all size-2 communities
#   in one split and the model would never see certain sizes during training.
#
#   stratify_by_size: we split WITHIN each size group so that each size
#   appears proportionally in both train and test.
# =============================================================================

print("\n  Splitting into train / test (stratified by community size) ...")

# Group graphs by community size, then split each group
train_graphs, test_graphs = [], []

# Get all unique community sizes
unique_sizes = sorted(set(g.community_size for g in graph_list))

for size in unique_sizes:
    # Filter graphs of this size
    size_graphs = [g for g in graph_list if g.community_size == size]
    random.shuffle(size_graphs)

    n_test = max(1, int(len(size_graphs) * TEST_FRAC))  # at least 1 test per size
    # max(1, ...): ensure we always have at least 1 test sample per size

    test_graphs  += size_graphs[:n_test]
    train_graphs += size_graphs[n_test:]

# Shuffle both splits so batches have mixed sizes
random.shuffle(train_graphs)
random.shuffle(test_graphs)

print(f"  Train: {len(train_graphs)} graphs")
print(f"  Test : {len(test_graphs)} graphs")
print(f"  Train size distribution: { {s: sum(1 for g in train_graphs if g.community_size==s) for s in unique_sizes} }")

# DataLoader: batches graphs together for efficient GPU processing
train_loader = GeoLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = GeoLoader(test_graphs,  batch_size=BATCH_SIZE, shuffle=False)

# =============================================================================
# SECTION 6 — MODEL (UNCHANGED FROM SIMULATED VERSION)
# =============================================================================
# The GATSS class is completely unchanged.
# The model architecture does NOT need to change because:
#   - in_channels = 1: still 1 node feature per strain (Lum, now normalised)
#   - hidden_channels, heads, dropout: same
#   - output: sigmoid → [0,1], same as suppression range
# The normalisation in Section 3 is what makes this work.
# =============================================================================

class GATSS(nn.Module):
    """
    Graph Attention Network — Strain Suppression (GAT-SS)
    
    UNCHANGED from simulated version. Works on real data because:
    - Node features are still [n_nodes, 1] (normalised Lum)
    - Labels are still suppression ∈ [0, 1]
    - Graph structure (fully connected) is identical
    """

    def __init__(self, in_channels, hidden_channels, heads, dropout):
        super(GATSS, self).__init__()

        # GATConv Layer 1: in=1, out=hidden per head, concat → hidden*heads total
        self.conv1 = GATConv(
            in_channels, hidden_channels,
            heads=heads, concat=True, dropout=dropout, add_self_loops=True
        )

        # GATConv Layer 2: in=hidden*heads, out=hidden, single head
        self.conv2 = GATConv(
            hidden_channels * heads, hidden_channels,
            heads=1, concat=False, dropout=dropout, add_self_loops=True
        )

        self.bn1     = nn.BatchNorm1d(hidden_channels * heads)
        self.bn2     = nn.BatchNorm1d(hidden_channels)
        self.dropout = nn.Dropout(p=dropout)

        # MLP head: (mean_pool || max_pool) → hidden*2 → hidden → 1
        self.lin1 = nn.Linear(hidden_channels * 2, hidden_channels)
        self.lin2 = nn.Linear(hidden_channels, 1)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        # Round 1: attention message passing
        x = self.conv1(x, edge_index)   # [total_nodes, hidden*heads]
        x = self.bn1(x)
        x = F.elu(x)                    # ELU: handles negative attention scores
        x = self.dropout(x)

        # Round 2: refine
        x = self.conv2(x, edge_index)   # [total_nodes, hidden]
        x = self.bn2(x)
        x = F.elu(x)

        # Pool all nodes → one vector per graph
        # mean: average strain behaviour; max: dominant strain signal
        x_mean = global_mean_pool(x, batch)           # [n_graphs, hidden]
        x_max  = global_max_pool(x, batch)            # [n_graphs, hidden]
        x = torch.cat([x_mean, x_max], dim=1)         # [n_graphs, hidden*2]

        # MLP → single suppression score
        x = self.dropout(x)
        x = F.elu(self.lin1(x))
        x = torch.sigmoid(self.lin2(x))               # output in (0, 1)

        return x.squeeze(-1)                          # [n_graphs]

    def get_node_embeddings(self, data):
        """
        Returns node embeddings after both GATConv layers.
        Used for PCA visualisation and Shapley computation.
        """
        x, edge_index = data.x, data.edge_index
        x = F.elu(self.bn1(self.conv1(x, edge_index)))
        x = F.elu(self.bn2(self.conv2(x, edge_index)))
        return x   # [n_nodes, hidden]


model = GATSS(
    in_channels     = 1,    # 1 normalised Lum value per strain
    hidden_channels = 32,
    heads           = 4,
    dropout         = 0.3,
)

print(f"\n  Model architecture:")
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Total trainable parameters: {total_params:,}")

# =============================================================================
# SECTION 7 — TRAINING (UNCHANGED except one addition)
# =============================================================================
# WHAT IS DIFFERENT:
#   Everything is the same EXCEPT we save the best model checkpoint.
#   Real lab data has fewer communities (1750 vs 1500 simulated) but the
#   training dynamics are similar.
#   We added early stopping + best model saving because real data can be
#   noisier and training for the full 300 epochs might overfit slightly.
# =============================================================================

print("\n  Training ...")

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
loss_fn   = nn.HuberLoss(delta=0.2)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", patience=40, factor=0.5
)

train_losses, test_losses = [], []

# NEW: track best model so far
# If test loss at epoch 200 is 0.005 and spikes to 0.010 at epoch 250,
# we keep the weights from epoch 200, not 250.
best_test_loss  = float("inf")
best_state_dict = None
# float("inf") = positive infinity — any real loss will be less than this
# state_dict() = dictionary of {layer_name: weight_tensor} for all layers

for epoch in range(1, N_EPOCHS + 1):

    # ── Training phase ────────────────────────────────────────────────────
    model.train()   # enables dropout + batch norm training mode
    total_loss, n_batches = 0.0, 0

    for batch in train_loader:
        optimizer.zero_grad()           # clear gradients from previous step
        pred = model(batch)             # forward pass
        loss = loss_fn(pred, batch.y.squeeze())
        loss.backward()                 # compute gradients via backpropagation
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        # clip_grad_norm_: cap gradient magnitude to prevent exploding gradients
        # max_norm=1.0: if total gradient norm > 1.0, scale all gradients down
        optimizer.step()                # update weights
        total_loss += loss.item()       # .item() extracts Python float from tensor
        n_batches  += 1

    avg_train_loss = total_loss / n_batches

    # ── Validation phase ──────────────────────────────────────────────────
    model.eval()    # disables dropout, uses running stats for batch norm
    total_test_loss, n_test_batches = 0.0, 0

    with torch.no_grad():   # no_grad: skip gradient tracking → faster inference
        for batch in test_loader:
            pred = model(batch)
            loss = loss_fn(pred, batch.y.squeeze())
            total_test_loss += loss.item()
            n_test_batches  += 1

    avg_test_loss = total_test_loss / max(n_test_batches, 1)
    train_losses.append(avg_train_loss)
    test_losses.append(avg_test_loss)
    scheduler.step(avg_test_loss)

    # NEW: save best weights
    if avg_test_loss < best_test_loss:
        best_test_loss = avg_test_loss
        # copy() makes a deep copy of the state dict so it doesn't change
        # as training continues (Python dicts are mutable, tensors are references)
        best_state_dict = {k: v.clone() for k, v in model.state_dict().items()}

    if epoch % 50 == 0 or epoch == 1:
        print(f"  Epoch {epoch:>3}/{N_EPOCHS} | "
              f"Train: {avg_train_loss:.4f} | "
              f"Test: {avg_test_loss:.4f} | "
              f"LR: {optimizer.param_groups[0]['lr']:.6f}")

# NEW: restore best weights before evaluation
# This ensures evaluation uses the best-seen model, not the final epoch
print(f"\n  Restoring best model (test loss = {best_test_loss:.4f}) ...")
model.load_state_dict(best_state_dict)
# load_state_dict(): load a previously saved weight dictionary back into the model

# =============================================================================
# SECTION 8 — EVALUATION
# =============================================================================
# WHAT IS DIFFERENT:
#   We add per-plate analysis (did the model generalise across different plates?)
#   and per-size analysis adapted to the real size bins {2,4,6,8,12,16,24,48}.
#   The evaluation logic is otherwise identical to the simulated version.
# =============================================================================

print("\n  Evaluating ...")

model.eval()
all_preds, all_true, all_sizes, all_plates = [], [], [], []

with torch.no_grad():
    for data in test_graphs:
        # Process one graph at a time
        # batch_tensor of all zeros tells PyG this is a single graph
        batch_tensor = torch.zeros(data.x.shape[0], dtype=torch.long)
        d    = Data(x=data.x, edge_index=data.edge_index, batch=batch_tensor)
        pred = model(d).item()
        all_preds.append(pred)
        all_true.append(data.y.item())
        all_sizes.append(data.community_size)
        all_plates.append(data.plate_replicate)

all_preds  = np.array(all_preds)
all_true   = np.array(all_true)
all_sizes  = np.array(all_sizes)
all_plates = np.array(all_plates)

r2              = r2_score(all_true, all_preds)
mae             = mean_absolute_error(all_true, all_preds)
mse             = np.mean((all_preds - all_true) ** 2)
spearman_rho, spearman_p = spearmanr(all_true, all_preds)

# Baseline: always predict the training mean
train_mean    = np.mean([g.y.item() for g in train_graphs])
baseline1_r2  = r2_score(all_true, np.full_like(all_true, train_mean))

# Baseline: mean normalised Lum of members (additive individual baseline)
# For each test community, average the node features and rescale
baseline2_preds = []
for data in test_graphs:
    mean_feat = data.x.mean().item()  # mean of normalised Lum values
    # Convert back to suppression: un-normalise → compute suppression
    raw_lum = mean_feat * scaler_ss.scale_[0] + scaler_ss.mean_[0]
    rescaled = np.clip(1.0 - 10.0 ** raw_lum, 0, 1)
    baseline2_preds.append(rescaled)
baseline2_preds = np.array(baseline2_preds)
baseline2_r2    = r2_score(all_true, baseline2_preds)

print(f"\n  ┌── GAT-SS Real Data Results ({len(test_graphs)} test graphs) ───────┐")
print(f"  │  R²          : {r2:.4f}                                    │")
print(f"  │  MAE         : {mae:.4f}                                    │")
print(f"  │  MSE         : {mse:.4f}                                    │")
print(f"  │  Spearman ρ  : {spearman_rho:.4f}  (p={spearman_p:.3e})             │")
print(f"  ├── Baselines ──────────────────────────────────────────────────┤")
print(f"  │  Global mean R²   : {baseline1_r2:.4f}                            │")
print(f"  │  Mean ss lum R²   : {baseline2_r2:.4f}                            │")
print(f"  └────────────────────────────────────────────────────────────────┘")

# NEW: per-plate R² (did we generalise across plate batches?)
print(f"\n  R² per plate replicate:")
for p in sorted(set(all_plates)):
    mask = all_plates == p
    if mask.sum() > 2:
        plate_r2 = r2_score(all_true[mask], all_preds[mask])
        print(f"    Plate {p}: R² = {plate_r2:.4f}  (n={mask.sum()})")

# =============================================================================
# SECTION 9 — SHAPLEY VALUES (UNCHANGED LOGIC, adapted baseline)
# =============================================================================
# WHAT IS DIFFERENT:
#   The compute_shapley_values function is IDENTICAL.
#   The only change: the baseline prediction (empty community) now uses
#   the training mean of the REAL lab data (not simulated data mean).
#   We also sample 30 communities but make sure we sample across all sizes.
# =============================================================================

print("\n  Computing Shapley values ...")

# The baseline prediction for an empty community = training mean suppression
# This represents "no information" — what we'd predict if no strains were present
baseline_pred = np.mean([g.y.item() for g in train_graphs])
print(f"  Baseline (empty community) prediction: {baseline_pred:.4f}")


def compute_shapley_values(model, data, n_permutations=N_SHAPLEY_PERMUTATIONS):
    """
    Permutation-based Shapley values for all nodes in one graph.
    
    For each random ordering of strains, we:
    1. Start from the baseline prediction (empty community)
    2. Add strains one by one in that order
    3. Record how much the prediction changes when each strain joins
    4. Average over many orderings → Shapley value per strain
    
    Positive value = this strain helps suppression
    Negative value = this strain hurts suppression (or is redundant)
    """
    n_nodes = data.x.shape[0]
    shapley_vals = np.zeros(n_nodes)

    for _ in range(n_permutations):
        # np.random.permutation: random ordering of node indices
        perm = np.random.permutation(n_nodes)
        prev_pred = baseline_pred   # start from the empty-community baseline

        for pos, node_idx in enumerate(perm):

            # Coalition = all nodes seen so far in this ordering
            coalition = perm[:pos + 1].tolist()

            # Build masked graph: nodes NOT in coalition get zeroed out
            # torch.zeros_like: creates tensor of zeros with same shape as data.x
            x_masked = torch.zeros_like(data.x)
            x_masked[coalition] = data.x[coalition]  # keep coalition nodes' features

            # Build a temporary Data object with masked features
            # We still pass ALL edges because the graph structure doesn't change —
            # the zeroed node features tell the model "these strains aren't here"
            masked_data = Data(x=x_masked, edge_index=data.edge_index)
            masked_data.batch = torch.zeros(n_nodes, dtype=torch.long)
            # batch of all zeros = treat all nodes as belonging to one graph

            with torch.no_grad():
                curr_pred = model(masked_data).item()

            # Marginal contribution: how much did adding node_idx change the prediction?
            marginal = curr_pred - prev_pred
            shapley_vals[node_idx] += marginal
            prev_pred = curr_pred

    # Average over all permutations
    shapley_vals /= n_permutations
    return shapley_vals


# Sample 30 communities for Shapley — pick across all size bins
print(f"  Computing Shapley values for 30 sample communities ...")
sample_graphs = []
for size in unique_sizes:
    size_test = [g for g in test_graphs if g.community_size == size]
    n_pick    = max(1, 30 // len(unique_sizes))  # roughly equal per size
    sample_graphs.extend(random.sample(size_test, min(n_pick, len(size_test))))

shapley_results = []
for data in sample_graphs:
    shap_vals = compute_shapley_values(model, data)
    for node_i, shap_val in enumerate(shap_vals):
        shapley_results.append({
            "community_id"   : data.community_id,
            "community_size" : data.community_size,
            "plate_replicate": data.plate_replicate,
            "strain_id"      : data.strain_ids[node_i],
            "lum_raw"        : data.x[node_i, 0].item() * scaler_ss.scale_[0] + scaler_ss.mean_[0],
            # undo normalisation to recover original Lum for plotting
            "shapley_value"  : shap_val,
            "abs_shapley"    : abs(shap_val),
        })

df_shap = pd.DataFrame(shapley_results)
print(f"  Shapley results: {len(df_shap)} strain-community pairs")
print(f"  Mean |Shapley|  : {df_shap['abs_shapley'].mean():.4f}")

# =============================================================================
# SECTION 10 — FIGURES (adapted for real data)
# =============================================================================

print("\n  Generating figures ...")

# ── Figure 1: Training curves ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("GAT-SS Real Data — Training Curves", fontsize=13, fontweight="bold")

epochs_x = range(1, len(train_losses) + 1)
axes[0].semilogy(epochs_x, train_losses, color=TEAL,   lw=2, label="Train")
axes[0].semilogy(epochs_x, test_losses,  color=ORANGE, lw=2, label="Test")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Huber Loss (log scale)")
axes[0].set_title("Loss Curves (Huber Loss)"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(list(epochs_x)[-100:], train_losses[-100:], color=TEAL,   lw=2, label="Train")
axes[1].plot(list(epochs_x)[-100:], test_losses[-100:],  color=ORANGE, lw=2, label="Test")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Huber Loss")
axes[1].set_title("Final 100 Epochs (zoomed)"); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig1_training_curves_ssr.png"), dpi=150, bbox_inches="tight")
plt.close(fig)
print("  → fig1_training_curves.png")

# ── Figure 2: Predicted vs Actual ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("GAT-SS Real Data — Prediction Performance", fontsize=13, fontweight="bold")

sc = axes[0].scatter(all_true, all_preds, c=all_sizes, cmap="viridis",
                     s=40, alpha=0.6, edgecolors="none")
plt.colorbar(sc, ax=axes[0], label="Community size (N)")
lims = [min(all_true.min(), all_preds.min()) - 0.02,
        max(all_true.max(), all_preds.max()) + 0.02]
axes[0].plot(lims, lims, "k--", lw=1.5, label="Perfect prediction")
axes[0].set_xlabel("Actual suppression"); axes[0].set_ylabel("Predicted suppression")
axes[0].set_title(f"Predicted vs Actual\nR²={r2:.3f}  MAE={mae:.3f}  ρ={spearman_rho:.3f}")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

models_r2 = [baseline1_r2, baseline2_r2, r2]
labels    = ["Global mean\n(Baseline 1)", "Mean Lum\n(Baseline 2)", "GAT-SS\n(Our model)"]
colors    = [GREY, GREY, TEAL]
bars = axes[1].bar(labels, models_r2, color=colors, width=0.5, edgecolor="white")
axes[1].axhline(0, color="black", lw=0.8)
for bar, val in zip(bars, models_r2):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + (0.01 if val >= 0 else -0.03),
                 f"{val:.3f}", ha="center", va="bottom", fontweight="bold", fontsize=10)
axes[1].set_ylabel("R² Score"); axes[1].set_title("R² Comparison: GAT-SS vs Baselines")
axes[1].grid(True, axis="y", alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig2_prediction_performance_ssr.png"), dpi=150, bbox_inches="tight")
plt.close(fig)
print("  → fig2_prediction_performance.png")

# ── Figure 3: Performance by size bin ────────────────────────────────────
# ADAPTED: use the actual lab sizes {2,4,6,8,12,16,24,48} as x-axis labels
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("GAT-SS Real Data — Performance vs Community Size", fontsize=13, fontweight="bold")

size_labels = sorted(unique_sizes)
maes_by_size, counts_by_size = [], []
for sz in size_labels:
    mask = all_sizes == sz
    if mask.sum() > 0:
        maes_by_size.append(mean_absolute_error(all_true[mask], all_preds[mask]))
        counts_by_size.append(mask.sum())
    else:
        maes_by_size.append(0); counts_by_size.append(0)

axes[0].bar([str(s) for s in size_labels], maes_by_size, color=TEAL, alpha=0.8, edgecolor="white")
for i, (mae_val, n) in enumerate(zip(maes_by_size, counts_by_size)):
    axes[0].text(i, mae_val + 0.001, f"n={n}", ha="center", va="bottom", fontsize=8, color=GREY)
axes[0].set_xlabel("Community size"); axes[0].set_ylabel("Mean Absolute Error")
axes[0].set_title("MAE per Community Size\n(lower = better)"); axes[0].grid(True, axis="y", alpha=0.3)

# R² per size bin
r2s_by_size = []
for sz in size_labels:
    mask = all_sizes == sz
    if mask.sum() > 2:
        r2s_by_size.append(r2_score(all_true[mask], all_preds[mask]))
    else:
        r2s_by_size.append(float("nan"))

axes[1].bar([str(s) for s in size_labels], r2s_by_size, color=TEAL, alpha=0.8, edgecolor="white")
axes[1].axhline(0, color="black", lw=0.8)
axes[1].set_xlabel("Community size"); axes[1].set_ylabel("R²")
axes[1].set_title("R² per Community Size"); axes[1].grid(True, axis="y", alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig3_performance_by_size_ssr.png"), dpi=150, bbox_inches="tight")
plt.close(fig)
print("  → fig3_performance_by_size.png")

# ── Figure 4: Shapley analysis ────────────────────────────────────────────
# ADAPTED: bins = actual lab sizes; x-axis labels are size values not ranges
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("GAT-SS Real Data — Shapley Value Analysis\n(strain-level contribution to community prediction)",
             fontsize=13, fontweight="bold")

bin_shap = df_shap.groupby("community_size")["abs_shapley"].agg(["mean", "std"]).reset_index()
axes[0].plot(range(len(bin_shap)), bin_shap["mean"],
             color=TEAL, lw=2.5, marker="o", markersize=8,
             markerfacecolor="white", markeredgewidth=2, markeredgecolor=TEAL)
axes[0].fill_between(range(len(bin_shap)),
                     bin_shap["mean"] - bin_shap["std"],
                     bin_shap["mean"] + bin_shap["std"],
                     alpha=0.15, color=TEAL)
axes[0].set_xticks(range(len(bin_shap)))
axes[0].set_xticklabels(bin_shap["community_size"].astype(str), fontsize=9)
axes[0].set_title("A) Mean |Shapley| vs Community Size\n(should decrease — dilution with diversity)")
axes[0].set_xlabel("Community size"); axes[0].set_ylabel("Mean absolute Shapley")
axes[0].grid(True, alpha=0.3)

axes[1].scatter(df_shap["lum_raw"], df_shap["shapley_value"],
                c=df_shap["community_size"], cmap="viridis_r",
                s=30, alpha=0.5, edgecolors="none")
axes[1].axhline(0, color=GREY, lw=1.2, ls="--")
axes[1].axvline(0, color=GREY, lw=1.2, ls="--")
rho_ss, _ = spearmanr(df_shap["lum_raw"], df_shap["shapley_value"])
axes[1].set_title(f"B) Lum (raw) vs Shapley Value\nSpearman ρ={rho_ss:.3f}")
axes[1].set_xlabel("Lum raw (individual suppression)"); axes[1].set_ylabel("Shapley value")
axes[1].grid(True, alpha=0.3)

sns.violinplot(data=df_shap, x="community_size", y="shapley_value",
               color=TEAL, alpha=0.6, linewidth=1.2, inner="quartile", ax=axes[2])
axes[2].axhline(0, color=GREY, lw=1.2, ls="--")
axes[2].set_title("C) Shapley Distribution per Size\n(should narrow with more diversity)")
axes[2].set_xlabel("Community size"); axes[2].set_ylabel("Shapley value")
axes[2].tick_params(axis="x", labelsize=8); axes[2].grid(True, axis="y", alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig4_shapley_analysis_ssr.png"), dpi=150, bbox_inches="tight")
plt.close(fig)
print("  → fig4_shapley_analysis.png")

# ── Figure 5: Node embeddings + attention ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("GAT-SS Real Data — Node Embeddings & Attention",
             fontsize=13, fontweight="bold")

# PCA on largest test community
largest_graph = max(test_graphs, key=lambda g: g.community_size)
model.eval()
with torch.no_grad():
    emb = model.get_node_embeddings(largest_graph).numpy()  # [N, 32]

pca    = PCA(n_components=2)
# fit_transform: fit PCA model to data AND project to 2D in one step
coords  = pca.fit_transform(emb)
# lum_raw for colouring: undo normalisation
lum_raw = largest_graph.x[:, 0].numpy() * scaler_ss.scale_[0] + scaler_ss.mean_[0]

sc = axes[0].scatter(coords[:, 0], coords[:, 1], c=lum_raw,
                     cmap="RdYlGn", s=80, edgecolors="white", linewidths=0.5, alpha=0.85)
plt.colorbar(sc, ax=axes[0], label="Lum raw")
axes[0].set_title(f"A) Node Embeddings (PCA)\nN={largest_graph.community_size} community\n"
                  f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%) | "
                  f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
axes[0].grid(True, alpha=0.3)

# Attention weights: find a size-4 or size-6 community (small enough to see clearly)
small_candidates = [g for g in test_graphs if g.community_size in [4, 6]]
small_graph = random.choice(small_candidates) if small_candidates else test_graphs[0]
with torch.no_grad():
    _, attn = model.conv1(small_graph.x, small_graph.edge_index,
                          return_attention_weights=True)
    edge_index_attn, attn_weights = attn
    mean_attn = attn_weights.mean(dim=1).numpy()  # average across heads

n_small = small_graph.x.shape[0]
attn_matrix = np.zeros((n_small, n_small))
ei = edge_index_attn.numpy()
for k in range(len(mean_attn)):
    attn_matrix[ei[1, k], ei[0, k]] = mean_attn[k]

sns.heatmap(attn_matrix, ax=axes[1], cmap="Blues",
            xticklabels=[f"S{i}" for i in range(n_small)],
            yticklabels=[f"S{i}" for i in range(n_small)],
            linewidths=0.5, linecolor="#e0e0e0",
            cbar_kws={"label": "Attention weight"})
axes[1].set_title(f"B) Attention Weights (N={small_graph.community_size} community)\n"
                  f"Row i attends to Column j — brighter = more attention")
axes[1].set_xlabel("Source node"); axes[1].set_ylabel("Target node")

plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig5_embeddings_attention_ssr.png"), dpi=150, bbox_inches="tight")
plt.close(fig)
print("  → fig5_embeddings_attention.png")

# =============================================================================
# SECTION 11 — SAVE
# =============================================================================

model_path = os.path.join(OUTPUT_DIR, "gat_ss_realdata_model.pt")
torch.save(model.state_dict(), model_path)

shap_path = os.path.join(OUTPUT_DIR, "shapley_values_gat_ss_realdata.csv")
df_shap.to_csv(shap_path, index=False)

# Save the scaler so you can un-normalise features later
import pickle
with open(os.path.join(OUTPUT_DIR, "scaler_ss.pkl"), "wb") as f:
    pickle.dump(scaler_ss, f)

print(f"\n  Model saved       : {model_path}")
print(f"  Shapley saved     : {shap_path}")
print(f"  Scaler saved      : scaler_ss.pkl")

print("\n" + "=" * 65)
print("  GAT-SS REAL DATA — COMPLETE")
print("=" * 65)
print(f"  Final R²         : {r2:.4f}")
print(f"  Final MAE        : {mae:.4f}")
print(f"  Spearman ρ       : {spearman_rho:.4f}")
print(f"  Beats global mean: {'YES' if r2 > baseline1_r2 else 'NO'}")
print(f"  All outputs in   : {OUTPUT_DIR}")
print("=" * 65)

  GAT-SS REAL LAB DATA — Loading files
  Single-strain strains loaded : 100
  Lum range (raw)              : -1.0009 to -0.1433
  Community rows loaded (all 5 plates) : 1750
  Community sizes present              : [np.int64(2), np.int64(4), np.int64(6), np.int64(8), np.int64(12), np.int64(16), np.int64(24), np.int64(48)]
  Lum range (raw)                      : -0.9985 to 0.0681

  Suppression stats:
    mean = 0.8171
    std  = 0.1221
    min  = 0.0000
    max  = 0.8997

  Mean suppression by community size:
n_strains
2     0.684
4     0.773
6     0.816
8     0.834
12    0.847
16    0.851
24    0.864
48    0.871
Name: community_suppression, dtype: float64

  Normalised node features:
    mean = 0.0000  (should be ≈ 0)
    std  = 1.0000   (should be ≈ 1)

  Building graphs ...
  Graphs built: 1750
  Sample graph — nodes: 2, features per node: 1, edges: 2
  Sample node features (first 3): [1.3562443256378174, 1.0529201030731201]
  Sample label: 0.0000

  Splitting into train / test (st